# Urine Marking Elo rating Calculation

This notebook takes in an input folder with excel sheets following the urine marking excel sheet template. 

It will create an excel sheet of given name in the directory defined as output_file_path. The excel sheet will contain a single sheet for final elo score and a sheet per file in the input folder of each individual match and elo score update. 

This notebook also has options for whether or not you have calculated ties before running this code or not. 

This also assumes all your columns headings are on the second line of the excel sheet if this is not true, please change header to 0.

In [1]:
import os
os.chdir(r'c:\\Users\\megha\\Documents\\GitHub\\social_competiton_elo_rating')
from collections import Counter
from collections import defaultdict
import pandas as pd
from src.elorating import calculation_edit as calculation
import numpy as np
import math

In [2]:
def write_excel(output_file_path, final_elo_df, elo_dfs):
    with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
        
        # Write the final ELO scores as the first sheet
        final_elo_df.to_excel(writer, sheet_name='Final Elo Scores', index=False)
        print(f"Written sheet: 'Final Elo Scores' with {len(final_elo_df)} rows")
        
        # Write each file's ELO dataframe as separate sheets
        for file_name, elo_df in elo_dfs.items():
            # Clean the sheet name
            sheet_name = file_name.replace('.xlsx','')
            
            # Handle potential duplicate sheet names
            original_sheet_name = sheet_name
            counter = 1
            while sheet_name in [ws.title for ws in writer.book.worksheets]:
                sheet_name = f"{original_sheet_name}_{counter}"
                if len(sheet_name) > 31:
                    sheet_name = f"{original_sheet_name[:28]}_{counter}"
                counter += 1
            
            # Write the dataframe to the sheet
            elo_df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"Written sheet: '{sheet_name}' with {len(elo_df)} rows")
def calculate_tie(file_df, percent_diff, min_spots, spot_columns):
    
    diff = abs(file_df[spot_columns[0]] - file_df[spot_columns[1]])
    spot_sum = file_df[spot_columns[0]] + file_df[spot_columns[1]]
    condition1 = (diff/(spot_sum/2) >= percent_diff)
    condition2 = diff >= min_spots
    file_df['ties'] = np.where(condition1 & condition2, 'NA', 'TIE')




In [6]:
#input folder to all the excel sheets with urinemarking data
input_folder = r"C:\Users\megha\Documents\GitHub\social_competiton_elo_rating\jupyter_notebooks\data\nina\urinemarking"

winner_column = 'winner' #label of the winner column exactly as it apperas in your excel sheets
loser_column = 'loser' #label of the loser column exactly as it apperas in your excel sheets
tie_column = 'ties' #label of the tie column exactly as it apperas in your excel sheets if it exists
tie_flag = 'TIE' #what is written in your excel to indicate that a match is a tie in your tie column
k_factor = 20 #k factor to use for elo rating calculations 

#if you have not calculated ties yet 
min_spots = 5 # min spot difference needed to not be considered a tie
percent_diff = 0.2 # min percent difference needed to not be considered a tie
spot_columns = ['left_number_of_spots', 'right_number_of_spots'] # name of spot number columns in your excel sheets

header = 1 #row of column names in your excel sheets, 0 indexed, 0 is top
#output folder and file name to save the elo rating excel sheet
output_file_path = r"C:\Users\megha\Documents\GitHub\social_competiton_elo_rating\jupyter_notebooks\data\nina\um_kfactor20_tryagain.xlsx"

In [ ]:

elo_dfs = {}
final_elo_dicts = {}
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith('.xlsx'):
            file_dfs = []
            raw_data_file_path = os.path.join(input_folder, file)
            xls = pd.ExcelFile(raw_data_file_path)
            for sheet in xls.sheet_names:
                per_sheet_dataframe = pd.read_excel(raw_data_file_path, sheet_name=sheet, header=header)
                per_sheet_dataframe = per_sheet_dataframe.dropna(axis=1, how='all')  
                per_sheet_dataframe['excel_file'] = file
                file_dfs.append(per_sheet_dataframe)
            file_df = pd.concat(file_dfs, ignore_index=True)
            if not tie_column:
                calculate_tie(file_df, percent_diff=percent_diff, min_spots=min_spots, spot_columns=spot_columns)
            final_elo_dict = defaultdict(float)
            elo_df = calculation.get_elo_df(dataframe=file_df, winner_id_column=winner_column, loser_id_column=loser_column, tie_column=tie_column, tie_flag = tie_flag, k_factor=k_factor)
            elo_dfs[file] = elo_df
            for subject in elo_df['subject_id'].unique():
                latest_elo = calculation.get_latest_elo_for_subject(elo_df, subject)
                final_elo_dict[subject] = latest_elo
            final_elo_dicts[file] = final_elo_dict

final_elo_df = pd.DataFrame([
    {'file': file_name, 'subject': subject, 'final elo score': elo}
    for file_name, subjects_dict in final_elo_dicts.items()
    for subject, elo in subjects_dict.items()])        
write_excel(output_file_path, final_elo_df, elo_dfs)

        date     match  left_number_of_spots  right_number_of_spots  winner  \
0 2023-06-15  1.1vs1.2                    22                      2     1.1   
1 2023-06-15  1.3vs1.4                     3                      3     1.3   
2 2023-06-19  1.1vs1.3                    39                     45     1.3   
3 2023-06-19  1.2vs1.4                     4                     22     1.4   
4 2023-06-21  1.1vs1.4                    66                     64     1.1   

   loser  spot_number_difference  percent_difference ties  \
0    1.2                      20            1.666667  NaN   
1    1.4                       0            0.000000  TIE   
2    1.1                       6            0.142857  TIE   
3    1.2                      18            1.384615  NaN   
4    1.4                       2            0.030769  TIE   

                                excel_file  
0  CD1_Urine_Marking_Assay_edit_final.xlsx  
1  CD1_Urine_Marking_Assay_edit_final.xlsx  
2  CD1_Urine_Marking_As

c:\Users\megha\Documents\GitHub\social_competiton_elo_rating\elo_rating_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
